# 12.5 Related solution values

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

Sets, parameters and variables are the most obvious things to look at in interpreting
the solution of a linear program, but AMPL also provides ways of examining objectives,
bounds, slacks, dual prices and reduced costs associated with the optimal solution.

As we have shown in numerous examples already, AMPL distinguishes the various
values associated with a `model` component by use of "qualified" names that consist of a
variable or constraint identifier, a dot (.), and a predefined "suffix" string. For instance,
the upper bounds for the variable Make are called Make.ub, and the upper bound for
Make["coils",2] is written Make["coils",2].ub. (Note that the suffix comes
after the subscript.) A qualified name can be used like an unqualified one, so that
`display` Make.ub prints a `table` of upper bounds on the Make variables, while
`display` Make, Make.ub prints a list of the optimal values and upper bounds.

**Objective functions**

The name of the `objective` function (from a minimize or maximize declaration)
refers to the objective's value computed from the current values of the variables. This
name can be used to represent the optimal `objective` value in `display`, print, or
printf:

```ampl
ampl: print 100 * Total_Profit /
ampl?    sum {p in PROD, t in 1..T} revenue[p,t] * Sell[p,t];
65.37528084182735
```

If the current `model` declares several `objective` functions, you can refer to any of them,
even though only one has been optimized.
**Bounds and slacks**

The suffixes .lb and .ub on a variable denote its lower and upper bounds, while .slack
denotes the difference of a variable's value from its nearer bound. Here's an
example from [Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1):

```ampl
ampl: display Buy.lb, Buy, Buy.ub, Buy.slack;
:    Buy.lb      Buy   Buy.ub Buy.slack    :=
BEEF    2      2          10    0
CHK     2     10          10    0
FISH    2      2          10    0
HAM     2      2          10    0
MCH     2      2          10    0
MTL     2      6.23596    10    3.76404
SPG     2      5.25843    10    3.25843
TUR     2      2          10    0
;
```

The reported bounds are those that were sent to the solver. Thus they include not only the
bounds specified in >= and <= phrases of `var` declarations, but also certain bounds that
were deduced from the constraints by AMPL's presolve phase. Other suffixes `let` you
look at the original bounds and at additional bounds deduced by presolve; see the discussion
of presolve in Section 14.1 for details.

Two equal bounds denote a fixed variable, which is normally eliminated by presolve.
Thus in the planning `model` of [Figure 4-4](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-4), the constraint Inv[p,0] = inv0[p] fixes
the initial inventories:

```ampl
ampl: display {p in PROD} (Inv[p,0].lb,inv0[p],Inv[p,0].ub);
:     Inv[p,0].lb inv0[p] Inv[p,0].ub    :=
bands      10        10        10
coils       0         0         0
;
```

In the production-and-transportation `model` of [Figure 4-6](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-6), the constraint

```ampl
sum {i in ORIG} Trans[i,j,p] = demand[j,p]
```

leads presolve to `fix` three variables at zero, because demand["LAN","plate"] is
zero:

```ampl
ampl: display {i in ORIG}
ampl?    (Trans[i,"LAN","plate"].lb,Trans[i,"LAN","plate"].ub);
:    Trans[i,'LAN','plate'].lb Trans[i,'LAN','plate'].ub    :=
CLEV              0                         0
GARY              0                         0
PITT              0                         0
;
```

As this example suggests, presolve's adjustments to the bounds may depend on the `data`
as well as the structure of the constraints.

The concepts of bounds and slacks have an analogous interpretation for the constraints
of a `model`. Any AMPL constraint can be put into the standard form

```ampl
lower bound ≤ body ≤ upper bound
```

where the body is a sum of all terms involving variables, while the lower bound and
upper bound depend only on the `data`. The suffixes .lb, .body and .ub give the current
values of these three parts of the constraint. For example, in the diet `model` of
[Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1) we have the declarations

```ampl
subject to Diet_Min {i in MINREQ}:
  sum {j in FOOD} amt[i,j] * Buy[j] >= n_Min[i];
subject to Diet_Max {i in MAXREQ}:
  sum {j in FOOD} amt[i,j] * Buy[j] <= n_Max[i];
```

and the following constraint bounds:

```ampl
ampl: display Diet_Min.lb, Diet_Min.body, Diet_Min.ub;
:   Diet_Min.lb Diet_Min.body Diet_Min.ub    :=
A         700       1013.98     Infinity
B1          0        605        Infinity
B2          0        492.416    Infinity
C         700        700        Infinity
CAL     16000      16000        Infinity
;

ampl: display Diet_Max.lb, Diet_Max.body, Diet_Max.ub;
:   Diet_Max.lb Diet_Max.body Diet_Max.ub    :=
A     -Infinity     1013.98       20000
CAL   -Infinity    16000          24000
NA    -Infinity    43855.9        50000
;
```

Naturally, <= constraints have no lower bounds, and >= constraints have no upper
bounds; AMPL uses -Infinity and Infinity in place of a number to denote these
cases. Both the lower and the upper bound can be finite, if the constraint is specified with
two <= or >= operators; see Section 8.4. For an = constraint the two bounds are the
same.

The suffix .slack refers to the difference between the body and the nearer bound:

```ampl
ampl: display Diet_Min.slack;
Diet_Min.slack [*] :=
  A 313.978
 B1 605
 B2 492.416
  C   0
CAL   0
;
```

For constraints that have a single <= or >= operator, the slack is always the difference
between the expressions to the left and right of the operator, even if there are variables on
both sides. The constraints that have a slack of zero are the ones that are truly constraining
at the optimal solution.
**Dual values and reduced costs**

Associated with each constraint in a linear program is a quantity variously known as
the dual variable, marginal value or shadow price. In the AMPL command environment,
these dual values are denoted by the names of the constraints, without any qualifying
suffix. Thus for example in [Figure 4-6](../04/4_3_a_model_of_production_and_transportation.ipynb#fig-4-6) there is a collection of constraints named Demand:

```ampl
subject to Demand {j in DEST, p in PROD}:
  sum {i in ORIG} Trans[i,j,p] = demand[j,p];
```

and a `table` of the dual values associated with these constraints can be viewed by

```ampl
ampl: display      Demand;
Demand [*,*]
:     bands     coils  plate :=
DET   201      190.714  199
FRA   209      204      211
FRE   266.2    273.714  285
LAF   201.2    198.714  205
LAN   202      193.714    0
STL   206.2    207.714  216
WIN   200      190.714  198
;
```

Solvers return optimal dual values to AMPL along with the optimal values of the "primal
" variables. We have space here only to summarize the most common interpretation
of dual values; the extensive theory of duality and applications of dual variables can be
found in any textbook on linear programming.

To start with an example, consider the constraint Demand["DET","bands"]
above. If we change the value of the parameter demand["DET","bands"] in this
constraint, the optimal value of the `objective` function Total_Cost changes accordingly.
If we were to plot the optimal value of Total_Cost versus all possible values of
demand["DET","bands"], the result would be a cost "curve" that shows how overall
cost varies with demand for bands at Detroit.

Additional computation would be necessary to determine the entire cost curve, but
you can learn something about it from the optimal dual values. After you `solve` the linear
program using a particular value of demand["DET","bands"], the dual price for the
constraint tells you the slope of the cost curve, at the demand's current value. In our
example, reading from the `table` above, we find that the slope of the curve at the current
demand is 201. This means that total production and shipping cost is increasing at the
rate of $201 for each extra ton of bands demanded at DET, or is decreasing by $201 for
each reduction of one ton in the demand.

As an example of an inequality, consider the following constraint from the same
`model`:

```ampl
subject to Time {i in ORIG}:
  sum {p in PROD} (1/rate[i,p]) * Make[i,p] <= avail[i];
```

```ampl
optimal
		objective

	   constant term
```

**Figure 12-1:** Piecewise-linear plot of `objective` function.

Here it is revealing to look at the dual values together with the slacks:

```ampl
ampl: display Time, Time.slack;
:        Time   Time.slack    :=
CLEV   -1522.86    0
GARY   -3040       0
PITT       0      10.5643
;
```

Where the slack is positive, the dual value is zero. Indeed, the positive slack implies that
the optimal solution does not use all of the time available at PITT; hence changing
avail["PITT"] somewhat does not affect the optimum. On the other hand, where the
slack is zero the dual value may be significant. In the case of GARY the value is -3040,
implying that the total cost is decreasing at a rate of $3040 for each extra hour available
at GARY, or is increasing at a rate of $3040 for each hour lost.

In general, if we plot the optimal `objective` versus a constraint's constant term, the
curve will be convex piecewise-linear (Figure 12-1) for a minimization, or concave
piecewise-linear (the same, but upside-down) for a maximization.

In terms of the standard form lower bound ≤ body ≤ upper bound introduced previously,
the optimal dual values can be viewed as follows. If the slack of the constraint is
positive, the dual value is zero. If the slack is zero, the body of the constraint must equal
one (or both) of the bounds, and the dual value pertains to the equaled bound or bounds.
Specifically, the dual value is the slope of the plot of the `objective` versus the bound, evaluated
at the current value of the bound; equivalently it is the rate of change of the optimal
`objective` with respect to the bound value.

A nearly identical analysis applies to the bounds on a variable. The role of the dual
value is played by the variable's so-called reduced cost, which can be viewed from the
AMPL command environment by use of the suffix .rc. As an example, here are the
bounds and reduced costs for the variables in [Figure 5-1](../05/5_3_set_operations.ipynb#fig-5-1):

```ampl
ampl: display Buy.lb, Buy, Buy.ub, Buy.rc;
:    Buy.lb      Buy   Buy.ub    Buy.rc :=
BEEF    2      2          10     1.73663
CHK     2     10          10    -0.853371
FISH    2      2          10     0.255281
HAM     2      2          10     0.698764
MCH     2      2          10     0.246573
MTL     2      6.23596    10     0
SPG     2      5.25843    10     0
TUR     2      2          10     0.343483
;
```

Since Buy["MTL"] has slack with both its bounds, its reduced cost is zero.
Buy["HAM"] is at its lower bound, so its reduced cost indicates that total cost is increasing
at about 70 cents per unit increase in the lower bound, or is decreasing at about 70
cents per unit decrease in the lower bound. On the other hand, Buy["CHK"] is at its
upper bound, and its negative reduced cost indicates that total cost is decreasing at
about 85 cents per unit increase in the upper bound, or is increasing at about 85 cents per unit
decrease in the upper bound.

If the current value of a bound happens to lie right at a breakpoint in the relevant
curve — one of the places where the slope changes abruptly in Figure 12-1 — the `objective`
will change at one rate as the bound increases, but at a different rate as the bound
decreases. In the extreme case either of these rates may be infinite, indicating that the linear
program becomes infeasible if the bound is increased or decreased by any further
amount. A solver reports only one optimal dual price or reduced cost to AMPL, however,
which may be the rate in either direction, or some value between them.

In any case, moreover, a dual price or reduced cost can give you only one slope on the
piecewise-linear curve of `objective` values. Hence these quantities should only be used as
an initial guide to the objective's sensitivity to certain variable or constraint bounds. If
the sensitivity is very important to your application, you can make a series of runs with
different bound settings; see [Section 11.3](../11/11_3_modifying_data.ipynb) for ways to quickly change a small part of the
`data`. (There do exist algorithms for finding part or all of the piecewise-linear curve,
given a linear change in one or more bounds, but they are not directly supported by the
current version of AMPL.)